# OpenAI 호환 LLM API 연결 (OpenAI-Compatible Connectivity)
## OpenAI 호환 프로토콜 — NVIDIA build · OpenRouter 두 경로

> 📦 **환경 설치·실행 명령**은 [`env_guides/M02_1_local_llm.md`](env_guides/M02_1_local_llm.md) 에 정리되어 있습니다.
> 반복·공통 코드는 [`agentic_lib/`](agentic_lib) 라이브러리(bootstrap·tools)로 분리해 두었습니다.

---

### 전제조건
- Windows 11 + **CMD(`cmd.exe`)** + Python **3.11**(`uv` 관리), 커널 = **`Agentic AI (uv)`**
- 기본 LLM = **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`**, 비교 대상 = **OpenRouter + `openrouter/free`**
- `notebooks/.env` 에 `NVIDIA_API_KEY`, `OPENROUTER_API_KEY` 설정(발급 방법은 [`M02_0_free_llm_api.ipynb`](M02_0_free_llm_api.ipynb) 참고)
- 공통 1회 준비(uv 설치 → `uv venv` → `uv sync` → 커널 등록)는 [`env_guides/README.md`](env_guides/README.md) 참고

### 학습 목표
이 노트북은 `.env` 의 `LLM_PROVIDER` 설정과 **무관하게**, 두 클라우드 경로를 **각각 명시적으로** 실행합니다.
1. **경로 1 — NVIDIA build**: (1) 연결 테스트 · (2) OpenAI 호환 응답 테스트
2. **경로 2 — OpenRouter**: (1) 연결 테스트 · (2) OpenAI 호환 응답 테스트

두 서비스 모두 **OpenAI 호환 `/v1`** 을 제공하므로, `base_url`·키·모델만 바꿔 **동일한 `ChatOpenAI` 코드**로 테스트합니다.

### 아키텍처 개요
```
                        ┌─→ [경로 1] NVIDIA build  · integrate.api.nvidia.com/v1 · deepseek-v4.1-flash
[LangChain ChatOpenAI] ─┤
                        └─→ [경로 2] OpenRouter    · openrouter.ai/api/v1        · openrouter/free
        (둘 다 OpenAI 호환 /v1 — base_url 만 바꾸면 코드는 그대로)
```

In [ ]:
# [setup] 자기완결적 셋업 — 이 노트북만 단독으로 실행 가능하게 한다
import sys, os
sys.path.insert(0, os.path.abspath(''))   # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()   # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import get_llm, print_provider_status

# 반복/공통 코드는 agentic_lib 라이브러리로 분리 (이 노트북은 bootstrap 만 사용)
from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text   # 공급자 무관 응답 정규화(<think> 제거 포함)

# 두 경로 모두 OpenAI 호환 → langchain-openai 의 ChatOpenAI 하나로 접속
utils.uv_install(['langchain', 'langchain-openai', 'langchain-nvidia-ai-endpoints',
                  'openai', 'requests'])

print_provider_status()

## 1. 환경 설정 — OpenAI 호환 프로토콜

**OpenAI 호환 API** 란 OpenAI 가 정한 요청·응답 형식(`/v1/chat/completions`, `/v1/models` 등)을 그대로 따르는
API 입니다. NVIDIA build 와 OpenRouter 는 모두 이 형식을 제공하므로, LangChain 의 `ChatOpenAI` 에
`base_url`·`api_key`·`model` 만 바꿔 넣으면 **코드 변경 없이** 공급자를 오갈 수 있습니다.

### 이 노트북의 진행 방식 (`.env` 와 무관)
아래 §2·§3 에서 **두 경로를 각각 명시적으로** 실행합니다. `.env` 의 `LLM_PROVIDER` 값이 무엇이든,
경로별 `base_url`·키·모델을 **코드에서 직접 지정**하므로 설정에 좌우되지 않습니다.

| 경로 | 공급자 | `base_url` | 모델 | 특징 |
|---|---|---|---|---|
| **경로 1** | `nvidia` | `https://integrate.api.nvidia.com/v1` | `deepseek-ai/deepseek-v4.1-flash` | 무료 크레딧, 모델 고정 |
| **경로 2** | `openrouter` | `https://openrouter.ai/api/v1` | `openrouter/free` | 무료 모델 자동 라우팅 |

각 경로마다 **(1) 연결 테스트**(OpenAI 호환 `/v1/models`)와 **(2) 응답 테스트**(`ChatOpenAI.invoke`)를 수행합니다.

> 설치·문제해결은 [`env_guides/M02_1_local_llm.md`](env_guides/M02_1_local_llm.md) 참고.

In [ ]:
# [공통 헬퍼] OpenAI 호환 API 테스트 — (1) 연결 테스트 · (2) 프로토콜 응답 테스트
# 이 노트북은 .env 의 LLM_PROVIDER 와 무관하게, 아래 두 경로를 '각각 명시적으로' 실행한다:
#   · 경로 1: NVIDIA build — deepseek-ai/deepseek-v4.1-flash · integrate.api.nvidia.com/v1
#   · 경로 2: OpenRouter   — openrouter/free                 · openrouter.ai/api/v1
# 두 서비스 모두 OpenAI 호환 /v1 이라, base_url·키·모델만 바꿔 같은 코드로 테스트한다.
import requests
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 경로별 접속 정보 (.env 에서 읽은 키·모델 사용)
ENDPOINTS = {
    "nvidia": dict(base_url=utils.NVIDIA_BASE_URL or "https://integrate.api.nvidia.com/v1",
                   api_key=utils.NVIDIA_API_KEY, model=utils.NVIDIA_MODEL),
    "openrouter": dict(base_url=utils.OPENROUTER_BASE_URL,
                       api_key=utils.OPENROUTER_API_KEY, model=utils.OPENROUTER_MODEL),
}

def check_openai_server(base_url, api_key, timeout=15.0):
    """OpenAI 호환 /v1/models 로 헬스체크 + 제공 모델 목록을 반환한다."""
    try:
        resp = requests.get(f"{base_url}/models", timeout=timeout,
                            headers={"Authorization": f"Bearer {api_key}"})
        resp.raise_for_status()
        return True, [m["id"] for m in resp.json().get("data", [])]
    except Exception as e:
        return False, str(e)

def connection_test(provider):
    """(1) '연결 테스트' — OpenAI 호환 /v1/models 로 서버 응답과 설정 모델 제공 여부를 확인한다."""
    ep = ENDPOINTS[provider]
    print(f"[{provider}] 연결 테스트 → {ep['base_url']}/models")
    if not ep["api_key"]:
        print(f"  ❌ 키 미설정 — .env 에 {provider.upper()}_API_KEY 를 추가하세요.")
        return False
    ok, info = check_openai_server(ep["base_url"], ep["api_key"])
    if not ok:
        print(f"  ❌ 연결 실패: {info}")
        return False
    listed = ep["model"] in info
    print(f"  ✅ 서버 응답 OK — 제공 모델 {len(info)}개")
    print(f"  설정 모델 {ep['model']!r}: " + ("목록에 있음 ✅" if listed else "목록에 없음 ⚠️ (서비스 종료 여부 확인)"))
    return ok

def make_openai_compatible(provider, temperature=0):
    """경로별 base_url·키·모델로 ChatOpenAI 를 만든다 — 공급자가 달라도 코드는 동일하다."""
    ep = ENDPOINTS[provider]
    return ChatOpenAI(model=ep["model"], base_url=ep["base_url"], api_key=ep["api_key"],
                      temperature=temperature, max_tokens=1024, timeout=120)

def response_test(provider):
    """(2) 'OpenAI 호환 프로토콜 응답 테스트' — ChatOpenAI 로 실제 추론한다.

    provider 를 인자로 직접 지정하므로 .env 의 LLM_PROVIDER 와 무관하게 원하는 경로로 접속한다.
    """
    llm = make_openai_compatible(provider)
    print(f"[{provider}] 응답 테스트 · LLM 타입={type(llm).__name__} · base_url={ENDPOINTS[provider]['base_url']}")
    demo = llm.invoke([
        SystemMessage(content="당신은 간결하게 답하는 한국어 어시스턴트입니다."),
        HumanMessage(content="LLM 을 OpenAI 호환 API 로 쓰는 장점을 한 문장으로 설명해줘."),
    ])
    # response_metadata 의 model_name: 실제로 응답을 생성한 모델(라우터는 호출마다 달라질 수 있음)
    print(f"  응답 모델: {demo.response_metadata.get('model_name', '(정보 없음)')}")
    print(f"  응답: {to_text(demo.content)}")      # to_text: 공급자 무관 정규화(<think> 제거 등)
    return llm

print("헬퍼 준비 완료 — connection_test(provider) · response_test(provider)")

---
## 2. 경로 1 — NVIDIA build

첫 번째 경로는 **NVIDIA build**(build.nvidia.com)입니다. 무료 크레딧으로 다양한 모델을 쓸 수 있고,
**OpenAI 호환 엔드포인트**(`https://integrate.api.nvidia.com/v1`)를 제공합니다. 본 강의의 기본 모델
**`deepseek-ai/deepseek-v4.1-flash`** 는 응답이 빠르고 한 응답에서 여러 도구를 **병렬 호출**할 수 있습니다.

> 이 노트북에서는 OpenAI 호환성을 보여 주려고 `ChatOpenAI` 로 접속합니다. 다른 노트북의 `utils.get_llm('nvidia')` 는
> NVIDIA 전용 커넥터 `ChatNVIDIA` 를 돌려주지만, 둘 다 같은 `.invoke()` 인터페이스를 가집니다(§4 참고).

진행: **(1) 연결 테스트** → **(2) OpenAI 호환 응답 테스트**

In [ ]:
# [경로 1] (1) NVIDIA build 연결 테스트 — OpenAI 호환 /v1/models
connection_test("nvidia")

In [ ]:
# [경로 1] (2) NVIDIA build OpenAI 호환 프로토콜 응답 테스트 (deepseek-v4.1-flash)
nvidia_llm = response_test("nvidia")   # ChatOpenAI(base_url=integrate.api.nvidia.com/v1)

---
## 3. 경로 2 — OpenRouter

두 번째 경로는 **OpenRouter**(openrouter.ai)입니다. 여러 회사의 모델을 하나의 **OpenAI 호환 엔드포인트**
(`https://openrouter.ai/api/v1`)로 묶어 제공하는 **모델 라우터**입니다.

기본 모델 **`openrouter/free`**(Free Models Router)는 요청마다 사용 가능한 무료 모델을 자동으로 고릅니다.
특정 `:free` 모델을 고정하면 과부하(503)나 요청 한도(429)로 실패할 수 있지만, 라우터는 다른 무료 모델로
넘어가므로 더 안정적입니다. 대신 **호출마다 응답 모델이 바뀔 수 있다**는 점에 유의하세요(아래 `응답 모델` 출력).

> **사용 한도**: 무료 모델은 하루 50요청(크레딧 10 이상 충전 시 1,000요청)으로 제한됩니다.

진행: **(1) 연결 테스트** → **(2) OpenAI 호환 응답 테스트**

In [ ]:
# [경로 2] (1) OpenRouter 연결 테스트 — OpenAI 호환 /v1/models
connection_test("openrouter")

In [ ]:
# [경로 2] (2) OpenRouter OpenAI 호환 프로토콜 응답 테스트 (openrouter/free)
openrouter_llm = response_test("openrouter")   # ChatOpenAI(base_url=openrouter.ai/api/v1)

---
## 4. 정리 — 두 경로 비교 (OpenAI 호환 프로토콜)

| 경로 | 서비스 | `base_url` | 모델 | 특징 |
|---|---|---|---|---|
| 경로 1 | **NVIDIA build** | `integrate.api.nvidia.com/v1` | `deepseek-ai/deepseek-v4.1-flash` | 모델 고정 → 결과 재현성 |
| 경로 2 | **OpenRouter** | `openrouter.ai/api/v1` | `openrouter/free` | 자동 라우팅 → 가용성 |

두 경로 모두 **같은 `ChatOpenAI` 코드**(`make_openai_compatible(provider).invoke(...)`)로 접속했고,
차이는 **`base_url`·키·모델**뿐입니다. 이것이 **OpenAI 호환 프로토콜**의 핵심 이점입니다.
공급자별 전용 커넥터(`ChatNVIDIA` 등)를 쓰더라도 LangChain 인터페이스는 같으므로,
다른 노트북에서는 `utils.get_llm()` 과 `.env` 의 `LLM_PROVIDER` 한 줄로 공급자를 전환합니다.

In [ ]:
# [정리] 두 경로를 나란히 확인 — 연결 상태 + 설정 모델 제공 여부
print("=== 두 클라우드 LLM 경로 (OpenAI 호환 /v1) ===")
for prov in ["nvidia", "openrouter"]:
    ep = ENDPOINTS[prov]
    ok, info = check_openai_server(ep["base_url"], ep["api_key"])
    status = (f"✅ 모델 {len(info)}개 제공 · 설정 모델 {'있음' if ep['model'] in info else '없음 ⚠️'}"
              if ok else f"❌ {info}")
    print(f"  경로: {prov:11s} {ep['base_url']:38s} {ep['model']:34s} {status}")

# 공급자 전용 커넥터(utils.get_llm)도 같은 인터페이스인지 확인
print("\nutils.get_llm() 이 돌려주는 커넥터:")
for prov in ["nvidia", "openrouter"]:
    print(f"  get_llm({prov!r:12s}) → {type(get_llm(prov)).__name__}  (.invoke / .stream / .bind_tools 동일)")